# 🧮 NumPy Assignment (Colab-ready) — ФИО/группа/ID + дедлайны + автопроверка

**Как работать:**
1. Заполните данные студента и дедлайны: дату **начала** и **окончания** приёма.
2. Решите задачи (ячейки с TODO).
3. Запустите ячейку **Автопроверка** — она посчитает сырой балл и применит штраф за опоздание:
   - Штраф считается **после** дедлайна, пропорционально опозданию относительно окна `due - start`.
   - Если опоздание равно длине окна, финальный балл становится **0**.
4. Последняя строка последней ячейки — JSON:
   ```json
   {"name": "...", "group": "...", "assignment": "...", "score": N}


In [2]:
# Студент:
student_first = "Василий"    #@param {type:"string"}
student_last  = "Петров"  #@param {type:"string"}
student_group = "AI-101"  #@param {type:"string"}
assignment_id = "NP-00"   #@param {type:"string"}

# Дедлайны (ISO-формат предпочтителен: "YYYY-MM-DDTHH:MM±HH:MM" или "YYYY-MM-DD HH:MM")
# Примеры: "2025-09-01T09:00-04:00" или "2025-09-05 23:59"
start_at_iso = "2025-09-02T09:00-04:00"  #@param {type:"string"}
due_at_iso   = "2025-09-09T23:59-04:00"  #@param {type:"string"}

# Для теста можно переопределить момент сдачи (оставьте пустым, чтобы взять текущее время):
submission_time_override = ""            #@param {type:"string"}

from datetime import datetime, timezone

def _parse_dt(s: str):
    s = (s or "").strip()
    if not s:
        return None
    # 1) Пытаемся ISO c offset: YYYY-MM-DDTHH:MM±HH:MM
    try:
        return datetime.fromisoformat(s)
    except Exception:
        pass
    # 2) Пытаемся "YYYY-MM-DD HH:MM"
    for fmt in ("%Y-%m-%d %H:%M", "%Y-%m-%d"):
        try:
            # Если формат без timezone — считаем как UTC, чтобы было детерминировано
            return datetime.strptime(s, fmt).replace(tzinfo=timezone.utc)
        except Exception:
            continue
    raise ValueError(f"Не удалось распарсить дату/время: {s!r}. Используйте ISO, например 2025-09-01T09:00-04:00")

full_name = (student_first.strip() + " " + student_last.strip()).strip()
start_dt = _parse_dt(start_at_iso)
due_dt = _parse_dt(due_at_iso)
submission_dt = _parse_dt(submission_time_override) if submission_time_override.strip() else datetime.now(timezone.utc)

def _sec(td):
    return td.total_seconds()

def penalty_fraction(start_dt, due_dt, now_dt) -> float:
    """
    Доля штрафа за опоздание:
    - До дедлайна: 0
    - После дедлайна: min(1, (now - due) / (due - start))
    - Если окно <= 0: штраф 1 после due, иначе 0
      (то есть после дедлайна всё мгновенно обнуляется).
    """
    if not (start_dt and due_dt and now_dt):
        return 0.0
    window = _sec(due_dt - start_dt)
    if window <= 0:
        return 1.0 if now_dt > due_dt else 0.0
    late = max(0.0, _sec(now_dt - due_dt))
    return min(1.0, late / window)

pf = penalty_fraction(start_dt, due_dt, submission_dt)

print("Студент:", full_name, "| Группа:", student_group, "| ID:", assignment_id)
print("Окно сдачи: ", start_dt, "→", due_dt)
print("Момент сдачи:", submission_dt)
print(f"Penalty fraction (доля штрафа): {pf:.4f}  (0 — нет, 1 — обнуление)")

Студент: Иван Петров | Группа: AI-101 | ID: NP-00
Окно сдачи:  2025-09-02 09:00:00-04:00 → 2025-09-09 23:59:00-04:00
Момент сдачи: 2025-09-01 14:07:32.404306+00:00
Penalty fraction (доля штрафа): 0.0000  (0 — нет, 1 — обнуление)


In [3]:
import numpy as np

def _close(a, b, tol=1e-8):
    return np.allclose(a, b, atol=tol)

def _arr_equal(a, b):
    a = np.array(a)
    b = np.array(b)
    return a.shape == b.shape and np.array_equal(a, b)


## Задача 1 (10 баллов)
Создайте одномерный массив `arr1` со значениями от `0` до `9` включительно.

**Требуется:** переменная `arr1` (тип `np.ndarray`, shape = `(10,)`).


In [4]:
# TODO: создайте arr1 со значениями 0..9 включительно
# Ваш код ниже:
arr1 = None
# arr1 = np.arange(10)


## Задача 2 (15 баллов)
Реализуйте функцию `sum_of_squares(a)`, которая **векторизованно** возвращает сумму квадратов элементов массива `a`.


In [5]:
# TODO: реализуйте функцию без циклов
def sum_of_squares(a: np.ndarray) -> float:
    # Ваш код ниже:
    # return ...
    raise NotImplementedError


## Задача 3 (15 баллов)
Создайте матрицу `D` размера `5x5`, где элементы главной диагонали равны `1`,
а элементы побочной диагонали равны `2`.
**Если элемент лежит на пересечении диагоналей (центр матрицы), его значение — `3`** (1+2).

**Требуется:** переменная `D` (тип `np.ndarray`, shape = `(5,5)`).


In [6]:
# TODO: создайте матрицу D по описанию выше
D = None
# Пример направления мысли:
# D = np.zeros((5,5))
# np.fill_diagonal(D, 1)
# np.fill_diagonal(np.fliplr(D), 2)
# D[2, 2] = 3


## Задача 4 (15 баллов)
Реализуйте функцию `broadcast_op(x, y)`, принимающую `x` формы `(5,1)` и `y` формы `(1,4)`.
Нужно вернуть матрицу `M = x * y + x + y` без циклов (используется broadcasting).

**Требуется:** функция возвращает массив формы `(5,4)`.


In [7]:
# TODO
def broadcast_op(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    # Ваш код ниже:
    # return ...
    raise NotImplementedError


## Задача 5 (15 баллов)
Реализуйте функцию `mean_above_mean(z)`, которая возвращает **среднее значение** элементов массива `z`,
которые **строго больше** среднего по всему `z`.

**Подсказка:** используйте булеву маску `z > z.mean()`.


In [8]:
# TODO
def mean_above_mean(z: np.ndarray) -> float:
    # Ваш код ниже:
    # return ...
    raise NotImplementedError


## Задача 6 (20 баллов)
Реализуйте функцию `solve_linear(A, b)`, которая решает СЛАУ `A x = b` (без ручного обращения матриц).

**Требуется:** использовать `np.linalg.solve`.


In [9]:
# TODO
def solve_linear(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    # Ваш код ниже:
    # return ...
    raise NotImplementedError


## Задача 7 (10 баллов)
Создайте функцию `row2_rev()`, которая формирует массив из чисел от `1` до `24`,
преобразует его в матрицу формы `(4,6)`, затем **возвращает вторую строку в обратном порядке**.

Ожидаемый результат: `[12, 11, 10, 9, 8, 7]`.


In [10]:
# TODO
def row2_rev() -> np.ndarray:
    # Ваш код ниже:
    # return ...
    raise NotImplementedError


## ▶️ Автопроверка (+ штраф за опоздание)
1. Считает «сырой» балл за задачи (макс. 100).
2. Применяет штраф после дедлайна: `final = raw * (1 - min(1, lateness / (due - start)))`.
3. Печатает JSON последней строкой (`score` — уже со штрафом).


In [11]:
import json
import numpy as np
from datetime import datetime, timezone

# ---------- тестовые помощники ----------
def add_points(ok, pts, msg_ok, msg_fail):
    global total
    if ok:
        total += pts
        print(f"[+{pts:>2}] {msg_ok}")
    else:
        print(f"[ 0] {msg_fail}")

def _sec(td):
    return td.total_seconds()

def penalty_fraction(start_dt, due_dt, now_dt) -> float:
    if not (start_dt and due_dt and now_dt):
        return 0.0
    window = _sec(due_dt - start_dt)
    if window <= 0:
        return 1.0 if now_dt > due_dt else 0.0
    late = max(0.0, _sec(now_dt - due_dt))
    return min(1.0, late / window)

# ---------- автогрейдинг ----------
total = 0
max_points = 100

# T1 (10)
try:
    ok = isinstance(arr1, np.ndarray) and arr1.shape == (10,) and np.array_equal(arr1, np.arange(10))
    add_points(ok, 10, "T1 OK", "T1 FAIL: arr1 должен быть np.arange(10)")
except Exception as e:
    add_points(False, 10, "", f"T1 EXC: {e}")

# T2 (15)
try:
    a = np.array([1,2,3], dtype=float)
    ok1 = abs(sum_of_squares(a) - 14.0) < 1e-8
    b = np.arange(6).reshape(2,3)
    ok2 = abs(sum_of_squares(b) - np.sum(b**2)) < 1e-8
    add_points(ok1 and ok2, 15, "T2 OK", "T2 FAIL: проверьте векторизацию/возврат суммы")
except Exception as e:
    add_points(False, 15, "", f"T2 EXC: {e}")

# T3 (15)
try:
    ok = isinstance(D, np.ndarray) and D.shape == (5,5)
    if ok:
        expect = np.zeros((5,5))
        np.fill_diagonal(expect, 1)
        np.fill_diagonal(np.fliplr(expect), 2)
        expect[2,2] = 3
        ok = np.array_equal(D, expect)
    add_points(ok, 15, "T3 OK", "T3 FAIL: проверьте диагонали и пересечение (3)")
except Exception as e:
    add_points(False, 15, "", f"T3 EXC: {e}")

# T4 (15)
try:
    x = np.arange(1,6).reshape(-1,1)
    y = np.arange(1,5).reshape(1,-1)
    M = broadcast_op(x, y)
    ok = isinstance(M, np.ndarray) and M.shape == (5,4) and np.array_equal(M, x*y + x + y)
    add_points(ok, 15, "T4 OK", "T4 FAIL: broadcasting/формула/shape")
except Exception as e:
    add_points(False, 15, "", f"T4 EXC: {e}")

# T5 (15)
try:
    rng = np.random.default_rng(42)
    z = rng.normal(size=1000)
    expect = z[z > z.mean()].mean()
    got = mean_above_mean(z)
    ok = abs(float(got) - float(expect)) < 1e-8
    add_points(ok, 15, "T5 OK", "T5 FAIL: проверьте маску z > z.mean() и среднее")
except Exception as e:
    add_points(False, 15, "", f"T5 EXC: {e}")

# T6 (20)
try:
    A = np.array([[3.,1.,2.],[0.,1.,-1.],[2.,0.,1.]])
    b = np.array([1.,2.,3.])
    expect = np.linalg.solve(A, b)
    got = solve_linear(A, b)
    ok = isinstance(got, np.ndarray) and got.shape == (3,) and np.allclose(got, expect, atol=1e-10)
    add_points(ok, 20, "T6 OK", "T6 FAIL: используйте np.linalg.solve и верните вектор формы (3,)")
except Exception as e:
    add_points(False, 20, "", f"T6 EXC: {e}")

# T7 (10)
try:
    got = row2_rev()
    expect = np.array([12,11,10,9,8,7])
    ok = isinstance(got, np.ndarray) and got.shape == (6,) and np.array_equal(got, expect)
    add_points(ok, 10, "T7 OK", "T7 FAIL: ожидался [12 11 10 9 8 7]")
except Exception as e:
    add_points(False, 10, "", f"T7 EXC: {e}")

raw_score = float(total)

# применяем штраф
try:
    pf = penalty_fraction(start_dt, due_dt, submission_dt)
except NameError:
    # если кто-то запустил автопроверку до ячейки с дедлайнами
    from datetime import timezone
    pf = 0.0

final_score = max(0.0, raw_score * (1.0 - min(1.0, pf)))

print(f"Сырой балл: {raw_score}/{max_points}")
print(f"Штраф (доля): {pf:.4f}")
print(f"Итоговый балл после штрафа: {final_score:.2f}/{max_points}")

# Последняя строка — JSON, который читает harness
final = {
    "name": full_name,
    "group": student_group,
    "assignment": assignment_id,
    "score": float(final_score)
}



[ 0] T1 FAIL: arr1 должен быть np.arange(10)
[ 0] T2 EXC: 
[ 0] T3 FAIL: проверьте диагонали и пересечение (3)
[ 0] T4 EXC: 
[ 0] T5 EXC: 
[ 0] T6 EXC: 
[ 0] T7 EXC: 
Сырой балл: 0.0/100
Штраф (доля): 0.0000
Итоговый балл после штрафа: 0.00/100
{"name": "Иван Петров", "group": "AI-101", "assignment": "NP-00", "score": 0.0}


In [ ]:
print(json.dumps(final, ensure_ascii=False))